# GTSRB补充实验：DS模型的学习率敏感性

目的：给“模型设计与调优能力”补充真实实验依据。固定DS结构、训练种子42、数据划分、增强、batch_size和80轮预算，仅改变初始学习率0.01、0.05、0.1。

**实验边界**：原四种结构的测试结果已经看过，因此本Notebook只进行训练集/验证集补充分析。不会读取测试图片、重新评估测试集或替换原来的测试结论。三个种子的原研究与本次单种子实验分开统计。

**为什么重新跑0.05**：三个候选在同一环境从随机初始化独立训练，避免把旧Colab环境和新平台的差异混入学习率比较。旧的12组实验不用重跑，也不会被覆盖。

使用方法：
1. 在Kaggle导入本Notebook并启用可用GPU。
2. 添加原先使用的GTSRB公开数据集作为输入，须含`Train.csv`和`Train`图片目录；不用重新上传原工程或模型权重。
3. 按顺序运行。自动找到数据时，`DATA_ROOT`保持空字符串即可。
4. 完成后下载`GTSRB_LR_results.zip`，发回以便核验和补入报告。

中断：同一运行环境中重新运行会续训未完成组，并跳过已完成组。若平台会话被删除，必须先有保存的输出版本/压缩包备份；未导出的临时文件不能保证保留。Colab分支把结果写入Google Drive。


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile
import torch

assert torch.cuda.is_available(), "请先启用GPU；本Notebook不自动使用CPU训练。"
if Path('/kaggle/working').exists():
    BASE = Path('/kaggle/working')
elif Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/GTSRB_LR_Supplement_v1')
else:
    BASE = Path.cwd() / 'GTSRB_LR_Supplement_v1'
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / 'GTSRB_LR_tools'
OUT = BASE / 'GTSRB_LR_Sensitivity'
PROJECT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('工具目录:', PROJECT)
print('结果目录:', OUT)

EMBEDDED_ZIP = '...'
EXPECTED_ZIP_SHA256 = 'dc051d818571303e437f874809c70c0410df26527d79554ae02ca78ef3304458'

raw = base64.b64decode(EMBEDDED_ZIP)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_ZIP_SHA256
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for item in z.infolist():
        dest = (PROJECT / item.filename).resolve()
        assert dest.is_relative_to(PROJECT.resolve())
        contents = z.read(item.filename)
        if dest.exists() and dest.read_bytes() != contents:
            raise RuntimeError('已有工具文件与本Notebook不同，请使用新的结果目录：' + str(dest))
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(contents)
print('原训练代码和固定train/val清单已准备；未包含测试清单。')


## 挂载数据与可选的续训备份

`DATA_ROOT`为空时自动查找Train.csv。换实例时挂载`GTSRB_LR_results.zip`生成的私有Dataset；代码同时支持保留ZIP和Kaggle自动解压后的目录。恢复后先查看各组轮数，再启动训练。请勿使用原四结构研究的迁移包。

本次修订只增加已解压备份的查找和恢复，训练源码及实验设置保持原样。


In [ ]:
DATA_ROOT = ""  # 自动识别失败时再填写，如 /kaggle/input/某个数据集目录

def find_metadata(filename, roots, depth=5):
    found = []
    stack = [(Path(r), 0) for r in roots if Path(r).exists()]
    while stack:
        folder, level = stack.pop()
        try:
            for p in folder.iterdir():
                if p.is_file() and p.name == filename:
                    found.append(p)
                elif p.is_dir() and level < depth and p.name.lower() not in {
                    'train', 'test', 'meta', 'images', '.cache', 'runs'}:
                    stack.append((p, level + 1))
        except PermissionError:
            pass
    return sorted(set(found))

search_roots = ['/kaggle/input'] if Path('/kaggle/input').exists() else ['/content/gtsrb_data', '/content/drive/MyDrive']
if not DATA_ROOT:
    candidates = [p.parent for p in find_metadata('Train.csv', search_roots)]
    if len(candidates) != 1:
        raise RuntimeError('发现的数据根目录：' + repr([str(p) for p in candidates]) + '；请在DATA_ROOT中指定原GTSRB目录。')
    DATA_ROOT = str(candidates[0])
DATA = Path(DATA_ROOT).resolve()
print('数据目录:', DATA)

import shutil

backups = find_metadata('GTSRB_LR_results.zip', ['/kaggle/input']) if Path('/kaggle/input').exists() else []
if len(backups) > 1:
    raise RuntimeError('发现多个补充实验备份，只保留要恢复的那个版本。')
if backups and not (OUT / 'protocol.json').exists():
    with zipfile.ZipFile(backups[0]) as z:
        for item in z.infolist():
            rel = Path(item.filename)
            if not rel.parts or rel.parts[0] != OUT.name:
                continue
            dest = (BASE / rel).resolve()
            assert dest.is_relative_to(OUT.resolve())
            if item.is_dir():
                dest.mkdir(parents=True, exist_ok=True)
            else:
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(item.filename))
    print('恢复补充实验备份:', backups[0])
elif not (OUT / 'protocol.json').exists():
    # Kaggle may automatically expand uploaded zip archives into input folders.
    expanded = []
    for candidate in find_metadata('protocol.json', ['/kaggle/input']):
        try:
            spec = json.loads(candidate.read_text())
        except (OSError, ValueError):
            continue
        if spec.get('study') == 'post_hoc_validation_lr_sensitivity_v1' and (candidate.parent / 'runs').is_dir():
            expanded.append(candidate.parent)
    if len(expanded) > 1:
        raise RuntimeError('发现多个已解压补充备份，请只挂载需要恢复的版本。')
    if expanded:
        shutil.copytree(expanded[0], OUT, dirs_exist_ok=True)
        print('恢复已解压的补充实验备份:', expanded[0])

for lr in [0.01, 0.05, 0.1]:
    folder = OUT / 'runs' / ('ds_lr' + format(lr, 'g').replace('.', 'p') + '_s42')
    history = folder / 'history.csv'
    import csv
    count = 0
    if history.exists():
        with history.open(encoding='utf-8-sig', newline='') as stream:
            count = len(list(csv.DictReader(stream)))
    print(f'{folder.name}：{count}/80轮，续训检查点：{(folder / "last.pt").exists()}')

CMD = [sys.executable, '-u', str(PROJECT / 'lr_sensitivity.py'),
       '--root', str(DATA), '--out', str(OUT), '--device', 'cuda']
subprocess.run(CMD + ['--prepare-only'], check=True, cwd=PROJECT)
print('固定划分、训练图片路径、源码和环境核验通过。计划：3组 × 80轮，seed=42。')


## 运行三组训练

顺序为0.01、0.05、0.1，每组80轮，batch_size=128。重复运行这一格会跳过已完成组，并使用各组自己的last.pt继续未完成组。

**不能把旧权重改学习率后直接续训来代替这里的对照。**三个候选需要相同的随机初始化规则和训练预算。

In [ ]:
subprocess.run(CMD, check=True, cwd=PROJECT)
print('三组训练已完成。下一格查看汇总并生成结果备份。')


## 汇总结果和图表

比较最佳验证Accuracy、对应Macro-F1、验证Loss和最佳轮次。程序按预先约定的验证Accuracy规则选择本次候选；不预设哪个学习率一定最好。单种子结果不作显著性或跨种子最优结论。

In [ ]:
subprocess.run(CMD + ['--summarize-only'], check=True, cwd=PROJECT)
from IPython.display import display, Markdown, Image
result = json.loads((OUT / 'validation_summary.json').read_text())
if result['complete']:
    display(Markdown((OUT / '报告补充结果.md').read_text()))
    display(Image(filename=str(OUT / 'figures/lr_learning_curves.png')))
    display(Image(filename=str(OUT / 'figures/lr_validation_comparison.png')))
else:
    print('尚未全部完成：', json.loads((OUT / 'progress.json').read_text()))
print('请下载此文件备份并发回：', BASE / 'GTSRB_LR_results.zip')
print('Kaggle请从文件面板或保存版本的Outputs下载；不要停止会话后再寻找临时文件。')
